# Embeddings com BERT em Português — Aplicado a Tweets de Times de Futebol

Neste notebook, adaptamos o pipeline de **embeddings com BERT em português** para um conjunto real de **tweets sobre times de futebol brasileiros** (`tweets_tratados_final.csv`).

O objetivo continua o mesmo: transformar texto em **vetores numéricos** (embeddings) e, a partir deles, aplicar **similaridade**, **clusterização** e **visualização 2D**.  
A diferença é que agora temos:

- **1.299 tweets reais**, escritos por usuários do Twitter/X.
- **12 times** de futebol como rótulo natural (Flamengo, Santos, Corinthians, Grêmio, Vasco, Cruzeiro, Palmeiras, São Paulo, Botafogo, Internacional, Fluminense, Atlético Mineiro).
- Texto curto, com gírias, hashtags e ruído típico de redes sociais.

Comparamos três modelos baseados em BERT (**mBERT**, **BERTimbau** e **BERTugues**) e duas estratégias de **pooling** (`CLS` e `MEAN`), e investigamos uma pergunta concreta:

> **Os embeddings conseguem agrupar tweets pelo time mencionado, sem nenhum treinamento supervisionado?**


### Instalação de Bibliotecas Necessárias

Antes de começarmos a trabalhar com embeddings e modelos de linguagem, precisamos instalar algumas bibliotecas que não vêm por padrão no ambiente.

O comando abaixo instala:

- **transformers**: biblioteca da Hugging Face para carregar modelos de linguagem como BERT, GPT etc.
- **sentence-transformers**: implementações otimizadas de modelos para embeddings semânticos de sentenças.
- **scikit-learn**: pacote essencial para aprendizado de máquina em Python, incluindo funções de clustering (ex.: KMeans), redução de dimensionalidade (ex.: PCA) e métricas.


In [ ]:
!pip install transformers datasets sentence-transformers scikit-learn

### Importação das Bibliotecas

Nesta etapa, organizamos os **imports** necessários para o projeto.  
Eles estão agrupados por finalidade para facilitar a leitura e o entendimento:

- **Numérico e dados**
  - `numpy` e `pandas`: manipulação de vetores, matrizes e dataframes.
  - `torch`: utilizado para trabalhar com tensores e rodar os modelos de linguagem da Hugging Face (com GPU se disponível).

- **Transformers / Hugging Face**
  - `AutoTokenizer` e `AutoModel`: classes que permitem carregar facilmente diferentes modelos pré-treinados de linguagem e seus tokenizadores.

- **Scikit-learn**
  - `KMeans`: algoritmo de clustering (agrupamento não supervisionado).
  - `PCA`: técnica de redução de dimensionalidade para visualização ou pré-processamento.
  - `cosine_similarity`: cálculo da similaridade semântica entre embeddings.
  - `silhouette_score` e `adjusted_rand_score`: métricas para avaliar a qualidade dos clusters e compará-los com os rótulos reais (times).

- **Visualização**
  - `matplotlib.pyplot`: biblioteca clássica para visualização estática.
  - `plotly.express` e `plotly.graph_objects`: visualizações interativas, como scatterplots e heatmaps.

Por fim, definimos a variável `seed = 42`, que serve como **semente aleatória** para garantir reprodutibilidade em algoritmos estocásticos (como PCA ou KMeans).


In [ ]:
# Numérico e dados
import numpy as np
import pandas as pd
import torch

# Transformers / Hugging Face
from transformers import AutoTokenizer, AutoModel

# Scikit-learn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Visualização
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

seed = 42
np.random.seed(seed)

### Carregando e Explorando o Dataset

O dataset `tweets_tratados_final.csv` contém tweets em português coletados sobre times de futebol brasileiros, já com tratamento básico (remoção de URLs, normalização etc.).

**Colunas:**

- `Time`: o time de futebol associado ao tweet (12 valores possíveis).
- `Nome`: nome de exibição do usuário.
- `Usuario`: handle do usuário (`@...`).
- `Texto`: o **texto do tweet** — é o que vamos transformar em embeddings.
- `Data`: data e hora da publicação.
- `topicos`: hashtags extraídas (pode ser `NaN`).
- `Stop words`: stopwords detectadas no texto (pode ser `NaN`).

Se você está rodando no **Google Colab**, é possível fazer o upload do arquivo executando o bloco de upload abaixo.  
Se estiver rodando localmente (Jupyter, VSCode etc.), basta deixar o CSV na mesma pasta do notebook que o código vai encontrá-lo automaticamente.


In [ ]:
# --- Carregamento robusto: tenta upload no Colab, senão tenta caminho local ---
import os

CSV_NAME = "tweets_tratados_final.csv"

if not os.path.exists(CSV_NAME):
    try:
        # Estamos no Colab? -> abre janela de upload
        from google.colab import files  # type: ignore
        print("Faça o upload do arquivo 'tweets_tratados_final.csv'...")
        uploaded = files.upload()
        # garante o nome esperado
        for fname in uploaded.keys():
            if fname != CSV_NAME:
                os.rename(fname, CSV_NAME)
    except ImportError:
        raise FileNotFoundError(
            f"Arquivo '{CSV_NAME}' não encontrado. "
            "Coloque-o na mesma pasta do notebook ou rode no Colab para fazer upload."
        )

df = pd.read_csv(CSV_NAME)
print(f"Linhas: {len(df)}  |  Colunas: {list(df.columns)}")
df.head()

Linhas: 1299  |  Colunas: ['Time', 'Nome', 'Usuario', 'Texto', 'Data', 'topicos', 'Stop words']


,Time,Nome,Usuario,Texto,Data,topicos,Stop words
0,Fluminense,guimarãess,@guimaffc,"merece fluminense fudido 2019, 2009, 2017, 201...",2026-03-19 19:03:32,NaN,"tu, o, de"
1,Flamengo,𐙚 jessi • arirang,@yoonbg7,"flamengo 20h comeback arirang 01hr, army flame...",2026-03-19 19:03:30,NaN,"e, tá"
2,Flamengo,CFD - Futebol e Fantasies,@CFD_Oficial,": rossi; emerson royal, léo ortiz, léo pereira...",2026-03-19 19:03:30,"Flamengo, Remo","e, e, e, e, de, e, e"
3,Flamengo,DECO SRN,@Deco_SRN,flamengo remo escalados!,2026-03-19 19:03:27,NaN,e
4,Flamengo,Deivede,@Deivede73,indo ver flamengo remo porque nunca vou saber ...,2026-03-19 19:03:24,NaN,"e, eu, um, do, de, e, por, que, o, do, é, e, p..."


In [ ]:
# Distribuição de tweets por time
contagem_times = df["Time"].value_counts()
print("Tweets por time:")
print(contagem_times)
print(f"\nTotal de times: {df['Time'].nunique()}")

Tweets por time:
Time
Flamengo            414
Santos              230
Corinthians          97
Grêmio               95
Vasco                92
São Paulo            81
Palmeiras            75
Botafogo             60
Cruzeiro             59
Fluminense           55
Internacional        36
Atlético Mineiro      5
Name: count, dtype: int64

Total de times: 12


### Amostragem Estratificada por Time

Rodar **3 modelos BERT** com **2 estratégias de pooling** sobre **1.299 tweets** é viável, mas no Colab CPU pode ficar lento (cerca de 6 passagens completas pelos modelos).

Para manter o notebook **didático e rápido**, fazemos uma **amostragem estratificada por time**:

- Pegamos no máximo `N_POR_TIME` tweets por time (com `random_state=seed` para reprodutibilidade).
- Isso garante que **todos os 12 times estejam representados**, mesmo os menos populares (Atlético Mineiro tem só 5 tweets, por exemplo).
- O resultado é um conjunto balanceado, ideal para visualizar agrupamentos.

> **Dica:** se quiser rodar com o dataset inteiro, basta colocar `N_POR_TIME = None`. Se quiser uma demo ainda mais rápida, reduza para `N_POR_TIME = 10`.


In [ ]:
# Parâmetros de amostragem — ajuste conforme seu hardware/tempo
N_POR_TIME = 25   # número máximo de tweets por time (None = todos)

def amostra_estratificada(df, col_grupo, n_por_grupo, random_state):
    """Amostragem estratificada compatível com qualquer versão do pandas (1.x, 2.x, 3.x)."""
    partes = []
    for _, sub in df.groupby(col_grupo, sort=False):
        n = min(len(sub), n_por_grupo)
        partes.append(sub.sample(n=n, random_state=random_state))
    return pd.concat(partes, ignore_index=True)

if N_POR_TIME is None:
    df_amostra = df.copy()
else:
    df_amostra = amostra_estratificada(df, "Time", N_POR_TIME, random_state=seed)

# Embaralha (importante: K-Means e visualizações ficam mais legíveis com dados misturados)
df_amostra = df_amostra.sample(frac=1, random_state=seed).reset_index(drop=True)

# Lista de textos que vamos passar para os modelos BERT
tweets = df_amostra["Texto"].astype(str).tolist()
times  = df_amostra["Time"].tolist()

print(f"Tweets amostrados: {len(tweets)}")
print(f"Distribuição por time na amostra:")
print(df_amostra['Time'].value_counts())
print(f"\nExemplo de tweet:\n  Time : {times[0]}\n  Texto: {tweets[0][:200]}")

Tweets amostrados: 280
Distribuição por time na amostra:
Time
Flamengo            25
Vasco               25
Botafogo            25
Internacional       25
Palmeiras           25
Fluminense          25
Cruzeiro            25
Santos              25
Corinthians         25
Grêmio              25
São Paulo           25
Atlético Mineiro     5
Name: count, dtype: int64

Exemplo de tweet:
  Time : Flamengo
  Texto: q porra essa? bora acordar flamengo!


### Funções Auxiliares: Preparação e Codificação dos Tweets

Nesta parte, criamos funções essenciais para trabalhar com embeddings de modelos baseados em BERT.

1. **`get_device()`**  
   Verifica se há uma GPU disponível.  
   - Se sim, retorna `"cuda"`.  
   - Caso contrário, usa `"cpu"`.  
   Isso garante que o código rode de forma eficiente em diferentes ambientes (Colab, PC pessoal, servidor com GPU, etc.).

2. **`mean_pooling(last_hidden_state, attention_mask)`**  
   Implementa o *mean pooling* manualmente:
   - O BERT gera uma matriz `last_hidden_state` com as representações de cada token (dimensão `[B, T, H]`, onde *B* é o batch, *T* o número de tokens e *H* o tamanho do embedding).  
   - Para calcular a média apenas sobre os tokens válidos (sem contar *padding*), usamos a `attention_mask`.  
   - O resultado é um vetor por sentença, representando a média dos embeddings de seus tokens.

3. **`encode_sentences(model_name, sentences, pooling="cls", batch_size=32)`**  
   Função que transforma sentenças (no nosso caso, **tweets**) em embeddings numéricos:
   - Carrega o **tokenizer** e o **modelo** a partir do nome informado (`model_name`).  
   - Para cada lote de sentenças (`batch`):
     - Aplica o tokenizador.  
     - Passa os tokens pelo modelo, obtendo `last_hidden_state`.  
     - Extrai os embeddings da forma escolhida em `pooling`:  
       - `"cls"`: usa apenas o primeiro token especial `[CLS]`.  
       - `"mean"`: usa a média dos embeddings de todos os tokens válidos (via `mean_pooling`).  
   - Retorna uma matriz `[N, H]`, em que cada linha é o embedding de um tweet.

   Obs.: a função é decorada com `@torch.no_grad()`, evitando o cálculo de gradientes (não faremos *fine-tuning* aqui, apenas inferência).

4. **`reduce_2d(X, method="pca", random_state=seed)`**  
   Reduz a dimensionalidade dos embeddings para 2 componentes principais, permitindo **visualizações em 2D**.


### Pooling em Modelos BERT: CLS vs. Mean

Quando usamos o BERT (ou modelos derivados) para gerar embeddings de sentenças, precisamos decidir **como transformar a sequência de embeddings de tokens em um único vetor representativo**.  
Esse processo é chamado de **pooling**.

#### 1. O token especial `[CLS]`
- O BERT adiciona, no início de cada sequência, um **token especial chamado `[CLS]`** (*classification token*).  
- Durante o treinamento original, esse token foi ajustado para capturar uma **representação global da sentença**, já que era usado em tarefas de classificação.  
- Vantagem: rápido, já vem pronto. Limitação: pode não capturar nuances semânticas em textos mais longos ou ruidosos (como tweets!).

#### 2. Pooling por média (*mean pooling*)
- Calcula a **média dos embeddings de todos os tokens** (ignorando *padding* via `attention_mask`).  
- Tende a ser mais robusto para tarefas de **similaridade semântica**, especialmente em textos curtos como tweets onde cada palavra carrega informação.

#### Em tweets, especificamente
Tweets são curtos, cheios de gírias, hashtags e nomes próprios (jogadores, times). O `mean pooling` costuma se sair melhor aqui, mas vamos comparar empiricamente.


In [ ]:
def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

def mean_pooling(last_hidden_state, attention_mask):
    # last_hidden_state: [B, T, H]; attention_mask: [B, T]
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

@torch.no_grad()
def encode_sentences(model_name, sentences, pooling="cls", batch_size=32):
    """
    pooling: 'cls' ou 'mean'
    """
    device = get_device()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_vecs = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        outputs = model(**inputs)  # last_hidden_state: [B, T, H]
        if pooling == "cls":
            vecs = outputs.last_hidden_state[:, 0, :]                       # [B, H]
        elif pooling == "mean":
            vecs = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])  # [B, H]
        else:
            raise ValueError("pooling deve ser 'cls' ou 'mean'")
        all_vecs.append(vecs.detach().cpu().numpy())

    X = np.vstack(all_vecs)
    return X  # shape: [N, H]

def reduce_2d(X, method="pca", random_state=seed):
    if method.lower() == "pca":
        reducer = PCA(n_components=2, random_state=random_state)
        Z = reducer.fit_transform(X)
        expl = reducer.explained_variance_ratio_
        return Z, ("PCA", expl)
    else:
        raise ValueError("method deve ser 'pca'")

### Modelos Utilizados

Criamos um dicionário (`modelos`) que associa nomes amigáveis a identificadores de modelos hospedados na Hugging Face:

- **mBERT** (`bert-base-multilingual-cased`)  
  Modelo multilíngue treinado em 104 idiomas, incluindo português.  
  Bom para tarefas que envolvem múltiplas línguas, mas tende a ter desempenho inferior em português em comparação com modelos monolíngues.

- **BERTimbau** (`neuralmind/bert-base-portuguese-cased`)  
  Modelo específico para o português brasileiro, treinado em grandes corpora no idioma.  
  Geralmente apresenta melhor desempenho em tarefas de PLN em português — promissor para nossos tweets brasileiros.

- **BERTugues** (`ricardoz/BERTugues-base-portuguese-cased`)  
  Outra iniciativa de modelo monolíngue em português, com vocabulário e pesos ajustados para o idioma.

Vamos comparar como esses três modelos representam tweets reais sobre futebol brasileiro.


In [ ]:
modelos = {
    "mBERT":     "bert-base-multilingual-cased",
    "BERTimbau": "neuralmind/bert-base-portuguese-cased",
    "BERTugues": "ricardoz/BERTugues-base-portuguese-cased",
}

### Geração dos Embeddings

Agora utilizamos os modelos para transformar cada tweet em um **vetor numérico** (embedding).

- A chave do dicionário `embeds` é uma tupla `(modelo, pooling)` — por exemplo, `("BERTimbau", "MEAN")`.  
- O valor é uma matriz NumPy `[N, H]`, onde:
  - `N` = número de tweets na amostra.  
  - `H` = dimensão do embedding (geralmente 768).

> **Atenção:** este passo baixa três modelos do Hugging Face Hub na primeira execução (alguns minutos) e roda inferência sobre todos os tweets. Em CPU pode levar de 1 a 5 minutos dependendo do tamanho da amostra; em GPU, segundos.


In [ ]:
# gerar embeddings
embeds = {}  # dict[(modelo, pooling)] -> np.ndarray [N, H]

for nome_visivel, nome_hf in modelos.items():
    print(f"--> Codificando com {nome_visivel} ({nome_hf})...")
    X_cls  = encode_sentences(nome_hf, tweets, pooling="cls")
    X_mean = encode_sentences(nome_hf, tweets, pooling="mean")
    embeds[(nome_visivel, "CLS")]  = X_cls
    embeds[(nome_visivel, "MEAN")] = X_mean

print("\nShapes dos embeddings gerados:")
for k, v in embeds.items():
    print(f"  {k}: {v.shape}")

--> Codificando com mBERT (bert-base-multilingual-cased)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ricardoz/BERTugues-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ricardoz/BERTugues-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Shapes dos embeddings gerados:
  ('mBERT', 'CLS'): (280, 768)
  ('mBERT', 'MEAN'): (280, 768)
  ('BERTimbau', 'CLS'): (280, 768)
  ('BERTimbau', 'MEAN'): (280, 768)
  ('BERTugues', 'CLS'): (280, 768)
  ('BERTugues', 'MEAN'): (280, 768)


### Similaridade por Cosseno (heatmap) e Agrupamento (K-Means)

#### Heatmap
Calculamos a **similaridade cosseno** entre os embeddings (`cosine_similarity(X)`) para inspecionar proximidade semântica entre os tweets.

> ⚠️ **Adaptação importante:** o heatmap original mostrava 6×6 = 36 células, perfeitamente legíveis. Com 300 tweets teríamos 90.000 células — ilegível.  
> Solução: para o heatmap, usamos uma **sub-amostra menor** (`N_HEATMAP` tweets, escolhidos com diversidade de times). Para a clusterização e a projeção 2D, continuamos usando **todos os tweets da amostra**.

#### Clusterização com K-Means

**Decisão:** vamos usar `n_clusters=12`, que é exatamente o número de **times** no dataset. Isso permite uma comparação direta:

> *Os clusters formados pelo BERT correspondem aos times reais?*

Se sim, isso indicaria que os tweets de torcedores de um mesmo time têm assinatura semântica suficientemente forte para serem reconhecidos sem supervisão.


In [ ]:
def plot_similarity_heatmap_px(
    X,
    sentences=None,
    labels_extra=None,        # rótulos adicionais (ex.: time) para o hover
    title="Similaridade (cosseno)",
    cmap="Blues",
    mask_upper=False,
    vmin=0.0, vmax=1.0,
    cbar_label="Similaridade",
    width=800, height=700,
    fmt=".2f",
    text_font_size=9,
    text_font_color="black",
    xgap=1, ygap=1,
    show_text=True
):
    S = cosine_similarity(X)
    n = S.shape[0]
    labels = [f"S{i+1}" for i in range(n)]

    Z = S.astype(float).copy()
    if mask_upper:
        iu = np.triu_indices(n, k=1)
        Z[iu] = np.nan

    text_matrix = np.empty((n, n), dtype=object)
    text_matrix[:] = ""
    if show_text:
        for i in range(n):
            for j in range(n):
                if not np.isnan(Z[i, j]):
                    text_matrix[i, j] = f"{Z[i, j]:{fmt}}"

    customdata = np.empty((n, n), dtype=object)
    customdata[:] = ""
    for i in range(n):
        for j in range(n):
            if not np.isnan(Z[i, j]):
                s1 = sentences[i] if sentences is not None else labels[i]
                s2 = sentences[j] if sentences is not None else labels[j]
                t1 = f" [{labels_extra[i]}]" if labels_extra is not None else ""
                t2 = f" [{labels_extra[j]}]" if labels_extra is not None else ""
                customdata[i, j] = f"<b>{labels[i]}{t1}</b>: {s1}<br><b>{labels[j]}{t2}</b>: {s2}"

    fig = go.Figure(
        data=go.Heatmap(
            z=Z, x=labels, y=labels,
            zmin=vmin, zmax=vmax, colorscale=cmap,
            colorbar=dict(title=cbar_label),
            text=text_matrix, texttemplate="%{text}",
            textfont=dict(color=text_font_color, size=text_font_size),
            customdata=customdata,
            hovertemplate="%{customdata}<extra></extra>",
            xgap=xgap, ygap=ygap
        )
    )
    fig.update_layout(
        title=title, width=width, height=height,
        template="plotly_white",
        margin=dict(l=60, r=30, t=60, b=60),
    )
    fig.update_yaxes(autorange="reversed", scaleanchor="x", scaleratio=1)
    fig.show()
    return S, fig


def run_kmeans(X, n_clusters=12, random_state=seed):
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init='auto')
    labels = km.fit_predict(X)
    return labels


# --- Sub-amostra menor só para o heatmap ---
N_POR_TIME_HEATMAP = 2          # 2 tweets por time -> ~24 tweets, heatmap legível
idx_heatmap = []
for _, sub in df_amostra.groupby("Time", sort=False):
    n = min(len(sub), N_POR_TIME_HEATMAP)
    idx_heatmap.extend(sub.sample(n=n, random_state=seed).index.tolist())

# Recortes para o heatmap
tweets_hm = [tweets[i] for i in idx_heatmap]
times_hm  = [times[i]  for i in idx_heatmap]
# encurta o texto que aparece no hover
tweets_hm_short = [t[:120] + ("..." if len(t) > 120 else "") for t in tweets_hm]

# ===== Loop principal: heatmap + K-Means =====
resultados = []
N_CLUSTERS = 12  # = nº de times

for (modelo, pooling), X in embeds.items():
    # Heatmap em sub-amostra
    X_hm = X[idx_heatmap]
    plot_similarity_heatmap_px(
        X_hm,
        sentences=tweets_hm_short,
        labels_extra=times_hm,
        title=f"Similaridade ({modelo}, {pooling}) — sub-amostra de {len(idx_heatmap)} tweets",
        cmap="Blues",
        mask_upper=False,
        vmin=0.0, vmax=1.0,
        cbar_label="Similaridade",
        fmt=".2f",
        text_font_size=8,
        xgap=1, ygap=1,
        show_text=True,
    )
    # K-Means na amostra COMPLETA (não na sub-amostra do heatmap)
    cluster_labels = run_kmeans(X, n_clusters=N_CLUSTERS, random_state=seed)
    resultados.append(pd.DataFrame({
        "modelo": modelo,
        "pooling": pooling,
        "tweet": tweets,
        "time_real": times,
        "cluster": cluster_labels,
    }))

clusters_df = pd.concat(resultados, ignore_index=True)
print(f"\nclusters_df shape: {clusters_df.shape}")


clusters_df shape: (1680, 5)


### Resultado da Clusterização

A tabela abaixo mostra, para cada combinação `(modelo, pooling)`, o cluster atribuído pelo K-Means a cada tweet, junto com o **time real** (rótulo que o BERT **não viu**).

Cada linha contém:
- **modelo**: `mBERT`, `BERTimbau` ou `BERTugues`.  
- **pooling**: `CLS` ou `MEAN`.  
- **tweet**: o texto original.  
- **time_real**: o rótulo verdadeiro (ground truth).  
- **cluster**: o rótulo de cluster atribuído pelo K-Means (0 a 11).

Vamos olhar uma amostra:


In [ ]:
# Mostra 15 linhas aleatórias para inspeção
clusters_df.sample(15, random_state=seed)[["modelo", "pooling", "time_real", "cluster", "tweet"]]

,modelo,pooling,time_real,cluster,tweet
1603,BERTugues,MEAN,Santos,3,essas turminha dono páginas fizesse tanto baru...
482,mBERT,MEAN,Internacional,6,brasil vergonha internacional.
203,mBERT,CLS,Santos,3,essas turminha dono páginas fizesse tanto baru...
49,mBERT,CLS,Grêmio,3,começou arena grêmio grêmio x vitória
937,BERTimbau,MEAN,Corinthians,7,corinthians
481,mBERT,MEAN,Botafogo,10,millonario blooming botafogo
1668,BERTugues,MEAN,Palmeiras,6,chega olha gol merda perdeu kkkkkkkkkkkkk furo...
1307,BERTugues,CLS,Palmeiras,3,palmeiras quer fazer pressão arbitragem sozinh...
1625,BERTugues,MEAN,Internacional,0,"rubio mandou real: estreito ormuz, onde passa ..."
518,mBERT,MEAN,Palmeiras,7,incrível como palmeiras sempre vilão pra imprensa


### Análise: os clusters refletem os times?

Esta é a pergunta central deste notebook. Como temos **rótulos reais** (`time_real`), podemos avaliar quantitativamente se o agrupamento não-supervisionado feito pelo BERT bate com a divisão por times.

Usamos duas métricas:

1. **Silhouette Score** (em relação ao próprio cluster do K-Means):  
   Mede o quão **bem definidos** estão os clusters no espaço de embeddings.  
   - Varia de **-1 a 1**. Valores próximos de 1 indicam clusters bem separados.  
   - **Não usa `time_real`** — é uma medida intrínseca da estrutura dos embeddings.

2. **Adjusted Rand Index (ARI)** (em relação a `time_real`):  
   Compara duas partições — a do K-Means e a dos times reais.  
   - Varia de **~0 (aleatório) a 1 (perfeito)**. Valores negativos indicam pior que o acaso.  
   - **Usa `time_real` como ground truth**.

Em conjunto, essas métricas dizem:  
- Se o **silhouette é alto mas o ARI é baixo** → o BERT achou estrutura, mas ela não corresponde aos times (talvez agrupe por sentimento, tema do jogo, etc.).  
- Se **ambos são altos** → os embeddings capturam identidade de time.


In [ ]:
# Calcula métricas para cada (modelo, pooling)
linhas = []
for (modelo, pooling), X in embeds.items():
    # Recupera os labels desse subset
    sub = clusters_df[(clusters_df["modelo"] == modelo) & (clusters_df["pooling"] == pooling)]
    cluster_labels = sub["cluster"].values
    time_labels = sub["time_real"].values

    sil = silhouette_score(X, cluster_labels) if len(set(cluster_labels)) > 1 else float("nan")
    ari = adjusted_rand_score(time_labels, cluster_labels)

    linhas.append({
        "modelo": modelo,
        "pooling": pooling,
        "silhouette (intrínseco)": round(sil, 4),
        "ARI vs time_real": round(ari, 4),
    })

metricas_df = pd.DataFrame(linhas).sort_values("ARI vs time_real", ascending=False).reset_index(drop=True)
print("Métricas de qualidade dos clusters:")
metricas_df

Métricas de qualidade dos clusters:


,modelo,pooling,silhouette (intrínseco),ARI vs time_real
0,BERTimbau,CLS,0.0292,0.0980
1,mBERT,MEAN,0.0231,0.0943
2,BERTimbau,MEAN,0.0256,0.0782
3,BERTugues,MEAN,0.0176,0.0445
4,mBERT,CLS,0.0376,0.0287
5,BERTugues,CLS,0.0268,0.0100


#### Cross-tab: cluster × time real

A tabela cruzada mostra, para a melhor combinação `(modelo, pooling)` segundo o ARI, **quantos tweets de cada time caíram em cada cluster**.  
Idealmente, cada time deveria se concentrar em poucos clusters (e vice-versa).


In [ ]:
# Pega a melhor combinação
best = metricas_df.iloc[0]
best_modelo, best_pooling = best["modelo"], best["pooling"]
print(f"Melhor combinação por ARI: {best_modelo} + {best_pooling}")
print(f"  silhouette = {best['silhouette (intrínseco)']}  |  ARI = {best['ARI vs time_real']}\n")

best_sub = clusters_df[
    (clusters_df["modelo"] == best_modelo) &
    (clusters_df["pooling"] == best_pooling)
]

ct = pd.crosstab(best_sub["time_real"], best_sub["cluster"])
print("Cross-tab (linhas = times, colunas = clusters do K-Means):")
ct

Melhor combinação por ARI: BERTimbau + CLS
  silhouette = 0.029200000688433647  |  ARI = 0.098

Cross-tab (linhas = times, colunas = clusters do K-Means):


cluster,0,1,2,3,4,5,6,7,8,9,10,11
time_real,,,,,,,,,,,,
Atlético Mineiro,0,1,0,1,0,1,0,1,0,0,1,0
Botafogo,0,9,3,7,0,1,0,2,0,0,3,0
Corinthians,0,1,2,5,3,2,0,7,0,0,3,2
Cruzeiro,0,1,1,2,0,1,1,2,1,9,3,4
Flamengo,0,1,3,2,1,9,0,3,0,2,3,1
Fluminense,0,2,1,1,0,11,0,2,2,2,1,3
Grêmio,0,5,2,0,8,4,0,0,0,1,0,5
Internacional,1,1,1,0,0,1,4,1,2,7,0,7
Palmeiras,0,0,2,3,0,0,0,13,2,0,4,1


### Projeção 2D (PCA) — Coloração por Cluster

Reduzimos os embeddings de alta dimensão para **2 componentes principais** e visualizamos em um único painel:

- **Colunas**: modelos (`mBERT`, `BERTimbau`, `BERTugues`).
- **Linhas**: estratégias de pooling (`CLS`, `MEAN`).
- **Cores**: clusters atribuídos pelo K-Means (rodado nos embeddings originais, não nos reduzidos).

> Pontos próximos no gráfico = tweets com embeddings similares.  
> Mesma cor = K-Means colocou no mesmo grupo.


In [ ]:
def plot_scatter_embeddings(embeds, sentences, times, method="pca", random_state=seed):
    """
    Gera DataFrame 2D concatenando projeções por (modelo, pooling),
    incluindo o time real para coloração alternativa.
    """
    rows = []
    for (modelo, pooling), X in embeds.items():
        Z, meta = reduce_2d(X, method=method, random_state=random_state)
        df_tmp = pd.DataFrame({
            "x": Z[:, 0],
            "y": Z[:, 1],
            "modelo": modelo,
            "pooling": pooling,
            "tweet": sentences,
            "time_real": times,
        })
        rows.append(df_tmp)
    return pd.concat(rows, ignore_index=True)


# 1) Preparar dados 2D (PCA)
df2d = plot_scatter_embeddings(embeds, sentences=tweets, times=times, method="pca")

# 2) Anexar cluster (merge usando uma coluna com ID estável: o índice da amostra original)
df2d["idx_amostra"] = df2d.groupby(["modelo", "pooling"]).cumcount()
clusters_df["idx_amostra"] = clusters_df.groupby(["modelo", "pooling"]).cumcount()
df2d = df2d.merge(
    clusters_df[["modelo", "pooling", "idx_amostra", "cluster"]],
    on=["modelo", "pooling", "idx_amostra"],
    how="left"
)
df2d["cluster"] = df2d["cluster"].astype(str)

# 3) Texto curto para hover
df2d["tweet_short"] = df2d["tweet"].apply(lambda s: s if len(s) <= 120 else s[:120] + "...")

# 4) Ordens explícitas
model_order = ["mBERT", "BERTimbau", "BERTugues"]
pool_order  = ["CLS", "MEAN"]

# 5) Scatter colorido por CLUSTER
fig = px.scatter(
    df2d,
    x="x", y="y",
    color="cluster",
    facet_col="modelo",
    facet_row="pooling",
    facet_col_spacing=0.06,
    facet_row_spacing=0.10,
    category_orders={"modelo": model_order, "pooling": pool_order},
    hover_data={
        "tweet_short": True,
        "time_real": True,
        "cluster": True,
        "modelo": False, "pooling": False,
        "x": ":.3f", "y": ":.3f"
    },
    title="Embeddings 2D (PCA) — colorido por cluster (K-Means, k=12)"
)
fig.update_layout(
    template="plotly_white",
    legend_title_text="Cluster",
    margin=dict(l=40, r=20, t=60, b=40),
    height=700, width=1100,
)
fig.update_traces(marker=dict(size=6, line=dict(width=0)), opacity=0.8)
fig.show()

### Projeção 2D (PCA) — Coloração por **Time Real**

Mesma projeção, mas agora colorida pelo **time real** (rótulo que o BERT não viu).

Esta é a visualização mais reveladora do notebook:

- **Se você enxergar regiões de cor coerentes** (ex.: pontos verdes do Flamengo agrupados em um canto), os embeddings BERT capturaram identidade de time só pelo texto.
- **Se as cores estiverem misturadas** (caos arco-íris), o sinal de "time" no texto é fraco em comparação com outras dimensões semânticas (sentimento, evento, gírias compartilhadas...).

Compare com o gráfico anterior (colorido por cluster) — se as duas visualizações se parecem, o K-Means encontrou os times.


In [ ]:
# Paleta com 12 cores distintas (uma por time)
palette = px.colors.qualitative.Light24[:12]

fig = px.scatter(
    df2d,
    x="x", y="y",
    color="time_real",
    color_discrete_sequence=palette,
    facet_col="modelo",
    facet_row="pooling",
    facet_col_spacing=0.06,
    facet_row_spacing=0.10,
    category_orders={"modelo": model_order, "pooling": pool_order},
    hover_data={
        "tweet_short": True,
        "time_real": True,
        "cluster": True,
        "modelo": False, "pooling": False,
        "x": ":.3f", "y": ":.3f"
    },
    title="Embeddings 2D (PCA) — colorido por TIME REAL"
)
fig.update_layout(
    template="plotly_white",
    legend_title_text="Time",
    margin=dict(l=40, r=20, t=60, b=40),
    height=700, width=1200,
)
fig.update_traces(marker=dict(size=6, line=dict(width=0)), opacity=0.8)
fig.show()

## Conclusão

Adaptamos o pipeline de embeddings BERT em português para um cenário **realista**: tweets brasileiros sobre futebol.

O que olhar agora:

- **Tabela `metricas_df`**: qual `(modelo, pooling)` deu o maior **ARI vs time_real**? Esse é o BERT que melhor "ouve" o time pelo jeito do tweet.
- **Heatmaps**: tweets do mesmo time tendem a ter similaridade alta entre si (cores escuras no eixo da diagonal por blocos)?
- **Cross-tab**: existem times que se concentram fortemente em um cluster (ex.: tweets do Flamengo quase todos no cluster 7)? E times que ficam dispersos?
- **Scatter colorido por time**: a olho nu, dá pra enxergar regiões por cor?

#### Próximos passos sugeridos

- Variar `N_POR_TIME` para ver como a estabilidade dos clusters muda com mais dados.
- Experimentar `n_clusters` diferente de 12 (8, 16, 24) e usar **elbow method** ou **silhouette** para escolher.
- Substituir K-Means por **HDBSCAN** (density-based) ou **AgglomerativeClustering** com linkage `ward` — costumam funcionar melhor em embeddings de texto.
- Usar um modelo de **sentence-embeddings** dedicado (`paraphrase-multilingual-MiniLM-L12-v2` da `sentence-transformers`) — embeddings de sentença direto, sem precisar escolher pooling.
- Refinar com `topicos` ou `Stop words` como features adicionais.


### Pooling: teoria e evidências práticas

O artigo *"Enhancing Sentence Embedding with Generalized Pooling"* (Chen, Ling & Zhu, 2018) mostra que, embora o token `[CLS]` seja comum para construir a representação de toda a sentença, sua eficácia depende fortemente de como o modelo foi treinado (se foi ajustado para tarefas de sentença, etc.).

Quando o modelo não tem supervisão direta para que `[CLS]` reflita bem todo o conteúdo da frase ou quando se quer usar embeddings "genéricos" para muitas tarefas, `mean pooling` ou variantes de pooling com atenção tendem a gerar representações mais estáveis e informativas. Ou seja:

- `CLS` é prático, barato computacionalmente (já vem com o modelo), bom para tarefas específicas (classificação se ajustado).
- `Mean pooling` reduz viés de seleção de token especial, tende a capturar melhor o contexto geral da frase, especialmente em tarefas de similaridade ou quando o modelo não foi supervisionado para usar CLS.

Em **tweets**, observe se o `MEAN` consistentemente ganha do `CLS` na coluna `ARI vs time_real` da tabela `metricas_df`. Se sim, a teoria se confirma na prática.
